#Setting up Drive and Dagshub/MLflow

In [1]:

from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/ML_final')

import torch
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

import pandas as pd
import numpy as np

Mounted at /content/drive
torch: 2.11.0+cu128 | CUDA: True


In [2]:
!pip install -q dagshub mlflow

import dagshub
import mlflow

from mlflow import MlflowClient
from mlflow.entities import ViewType


dagshub.init(
    repo_owner="skupr23",
    repo_name="ml_final",
    mlflow=True,
)

EXPERIMENT_NAME = "PatchTST_Training"

if mlflow.active_run() is not None:
    mlflow.end_run()


client = MlflowClient()

matches = client.search_experiments(
    view_type=ViewType.ALL,
    filter_string=f"name = '{EXPERIMENT_NAME}'",
)

if matches:
    experiment = matches[0]

    if experiment.lifecycle_stage == "deleted":
        client.restore_experiment(experiment.experiment_id)
        print(
            f"Restored deleted experiment: "
            f"{EXPERIMENT_NAME} ({experiment.experiment_id})"
        )


mlflow.set_experiment(EXPERIMENT_NAME)

active_experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("Tracking URI :", mlflow.get_tracking_uri())
print("Experiment   :", active_experiment.name)
print("Experiment ID:", active_experiment.experiment_id)
print("Lifecycle    :", active_experiment.lifecycle_stage)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 102.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 131.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 132.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=13524eb6-999a-4e71-91ff-e921adb8dc69&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=18eb8281576d5af889621f5f2085b80702da5d26cca7f991043a2ddad039a840




Accessing as skupr23

Initialized MLflow to track repo "skupr23/ml_final"

Repository skupr23/ml_final initialized!

Tracking URI : https://dagshub.com/skupr23/ml_final.mlflow
Experiment   : PatchTST_Training
Experiment ID: 7
Lifecycle    : active


# Cell 2 - Load processed data (identical source as DLinear notebook)


In [3]:
train = pd.read_pickle('data/train_processed.pkl')
train = train.sort_values(['Store','Dept','Date'])
print(train['Date'].min(), train['Date'].max())

2010-02-05 00:00:00 2012-10-26 00:00:00


# Cell 3 - WMAE metric (same scoring function used everywhere)


In [4]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

# Cell 4 - Select series and build clean per-series arrays.


In [5]:

series_totals = (
    train.groupby(['Store','Dept'])['Weekly_Sales']
    .sum()
    .sort_values(ascending=False)
)
top_series = series_totals.index.tolist()

INPUT_LEN = 52
OUTPUT_LEN = 13
VAL_LEN = 39
MIN_LEN = INPUT_LEN + VAL_LEN + 10

series_data = {}
for store, dept in top_series:
    sub = train[(train['Store']==store) & (train['Dept']==dept)].sort_values('Date')
    if len(sub) < MIN_LEN:
        continue
    sub = sub.set_index('Date').asfreq('W-FRI')
    sub['Weekly_Sales'] = sub['Weekly_Sales'].interpolate().bfill().ffill()
    sub['IsHoliday'] = sub['IsHoliday'].astype(float).fillna(0.0).astype(bool)
    series_data[(store, dept)] = {
        'values': sub['Weekly_Sales'].values.astype(np.float32),
        'is_holiday': sub['IsHoliday'].values,
        'dates': sub.index
    }

keys_kept = list(series_data.keys())
print('Series successfully built:', len(keys_kept))

Series successfully built: 2873


# Cell 5 - Split each series into fit/val, then build (input, output) windows


In [6]:

fit_raw, val_raw = {}, {}
for key, d in series_data.items():
    vals = d['values']
    fit_raw[key] = vals[:-VAL_LEN]
    val_raw[key] = vals[-VAL_LEN:]

X_train, Y_train, X_monitor, Y_monitor = [], [], [], []
for key, fseries in fit_raw.items():
    n = len(fseries)
    max_start = n - INPUT_LEN - OUTPUT_LEN
    if max_start < 1:
        continue

    for start in range(max_start):
        X_train.append(fseries[start:start+INPUT_LEN])
        Y_train.append(fseries[start+INPUT_LEN:start+INPUT_LEN+OUTPUT_LEN])
    X_monitor.append(fseries[max_start:max_start+INPUT_LEN])
    Y_monitor.append(fseries[max_start+INPUT_LEN:max_start+INPUT_LEN+OUTPUT_LEN])

X_train = np.stack(X_train).astype(np.float32)
Y_train = np.stack(Y_train).astype(np.float32)
X_monitor = np.stack(X_monitor).astype(np.float32)
Y_monitor = np.stack(Y_monitor).astype(np.float32)

print('Train windows:', X_train.shape, '| Monitor windows:', X_monitor.shape)

Train windows: (111752, 52) | Monitor windows: (2873, 52)


# Cell 6 - PatchTST implementation in plain PyTorch.


In [7]:

import torch.nn as nn

PATCH_LEN = 13
STRIDE = 13
NUM_PATCHES = INPUT_LEN // PATCH_LEN

class RevIN(nn.Module):

    def __init__(self, eps=1e-5):
        super().__init__()
        self.eps = eps

    def forward(self, x):

        mean = x.mean(dim=1, keepdim=True)
        std = x.std(dim=1, keepdim=True) + self.eps
        x_norm = (x - mean) / std
        return x_norm, mean, std

class PatchTST(nn.Module):
    def __init__(self, input_len, output_len, patch_len, num_patches,
                 d_model=64, nhead=4, num_layers=3, dim_feedforward=256, dropout=0.1):
        super().__init__()
        self.revin = RevIN()
        self.patch_len = patch_len
        self.num_patches = num_patches

        self.patch_embed = nn.Linear(patch_len, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.head = nn.Linear(num_patches * d_model, output_len)

    def forward(self, x):

        x_norm, mean, std = self.revin(x)
        patches = x_norm.reshape(x.size(0), self.num_patches, self.patch_len)
        tokens = self.patch_embed(patches) + self.pos_embed
        encoded = self.encoder(tokens)
        flat = encoded.reshape(x.size(0), -1)
        out_norm = self.head(flat)
        out = out_norm * std + mean
        return out

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_patchtst = PatchTST(INPUT_LEN, OUTPUT_LEN, PATCH_LEN, NUM_PATCHES).to(device)
print(model_patchtst)

PatchTST(
  (revin): RevIN()
  (patch_embed): Linear(in_features=13, out_features=64, bias=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (head): Linear(in_features=256, out_features=13, bias=True)
)


# Cell 7 - Train with early stopping on the monitor set, same spirit as


In [8]:

from torch.utils.data import TensorDataset, DataLoader

X_train_t = torch.from_numpy(X_train)
Y_train_t = torch.from_numpy(Y_train)
X_monitor_t = torch.from_numpy(X_monitor).to(device)
Y_monitor_t = torch.from_numpy(Y_monitor).to(device)

train_loader = DataLoader(TensorDataset(X_train_t, Y_train_t), batch_size=1024, shuffle=True)

loss_fn = torch.nn.HuberLoss(delta=1.0)
optimizer = torch.optim.AdamW(model_patchtst.parameters(), lr=5e-4, weight_decay=1e-2)
N_EPOCHS = 30
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

best_val_loss = float('inf')
patience, patience_counter = 5, 0
best_state = None
best_epoch = 0
history = []

for epoch in range(N_EPOCHS):
    model_patchtst.train()
    train_loss_sum, n_batches = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model_patchtst(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_patchtst.parameters(), 0.5)
        optimizer.step()
        train_loss_sum += loss.item()
        n_batches += 1
    scheduler.step()

    model_patchtst.eval()
    with torch.no_grad():
        val_pred = model_patchtst(X_monitor_t)
        val_loss = loss_fn(val_pred, Y_monitor_t).item()

    train_loss = train_loss_sum / n_batches
    history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'lr': scheduler.get_last_lr()[0],
    })

    print(f"Epoch {epoch+1} - train_loss: {train_loss:.5f} - val_loss: {val_loss:.5f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model_patchtst.state_dict().items()}
        best_epoch = epoch + 1
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping.")
            break

epochs_trained = epoch + 1
model_patchtst.load_state_dict(best_state)

Epoch 1 - train_loss: 2018.30633 - val_loss: 2948.06934
Epoch 2 - train_loss: 1692.67488 - val_loss: 2719.90991
Epoch 3 - train_loss: 1614.60877 - val_loss: 2615.20947
Epoch 4 - train_loss: 1577.86549 - val_loss: 2586.22949
Epoch 5 - train_loss: 1546.92428 - val_loss: 2500.98901
Epoch 6 - train_loss: 1523.60348 - val_loss: 2501.84814
Epoch 7 - train_loss: 1507.62880 - val_loss: 2465.29419
Epoch 8 - train_loss: 1490.43059 - val_loss: 2417.23584
Epoch 9 - train_loss: 1480.59268 - val_loss: 2403.82544
Epoch 10 - train_loss: 1467.08141 - val_loss: 2411.19263
Epoch 11 - train_loss: 1454.56416 - val_loss: 2374.34155
Epoch 12 - train_loss: 1444.71530 - val_loss: 2367.58716
Epoch 13 - train_loss: 1438.28866 - val_loss: 2382.13940
Epoch 14 - train_loss: 1431.98060 - val_loss: 2362.17944
Epoch 15 - train_loss: 1423.57001 - val_loss: 2387.34961
Epoch 16 - train_loss: 1419.20365 - val_loss: 2344.46753
Epoch 17 - train_loss: 1412.81293 - val_loss: 2379.26318
Epoch 18 - train_loss: 1409.56207 - val_

<All keys matched successfully>

# Cell 8 - Predict 39 weeks ahead per series by rolling the 13-week output


In [9]:

model_patchtst.eval()
all_true, all_pred, all_holiday = [], [], []

with torch.no_grad():
    for key in keys_kept:
        context = fit_raw[key][-INPUT_LEN:].copy()
        true_vals = val_raw[key]
        preds = []
        for _ in range(VAL_LEN // OUTPUT_LEN):
            x = torch.from_numpy(context[-INPUT_LEN:]).unsqueeze(0).to(device)
            block = model_patchtst(x).cpu().numpy().flatten()
            preds.extend(block)
            context = np.concatenate([context, block])

        preds = np.array(preds[:VAL_LEN])
        all_true.extend(true_vals)
        all_pred.extend(preds)
        all_holiday.extend(series_data[key]['is_holiday'][-VAL_LEN:])

all_true = np.array(all_true)
all_pred = np.array(all_pred)
all_holiday = np.array(all_holiday)
print('Done. Total validation points:', len(all_true))

Done. Total validation points: 112047


# Cell 9 - Evaluate


In [10]:
patchtst_wmae = wmae(all_true, all_pred, all_holiday)
print('PatchTST (custom PyTorch) WMAE:', patchtst_wmae)
print('Holiday WMAE:    ', wmae(all_true[all_holiday], all_pred[all_holiday], all_holiday[all_holiday]))
print('Non-holiday WMAE:', wmae(all_true[~all_holiday], all_pred[~all_holiday], all_holiday[~all_holiday]))

PatchTST (custom PyTorch) WMAE: 1730.4815471269837
Holiday WMAE:     1825.8697027562484
Non-holiday WMAE: 1704.833347957983


# Cell 10 - Save model locally


In [11]:
import os
import joblib

os.makedirs('models', exist_ok=True)
joblib.dump(model_patchtst.cpu(), 'models/patchtst_best.pkl')
print('Saved.')

Saved.


In [12]:

import os
import tempfile

import cloudpickle
import mlflow.pytorch

PREPROCESSING_STEPS = [
    {'step': 'load_processed', 'detail': 'read data/train_processed.pkl, sort by Store-Dept-Date'},
    {'step': 'series_selection', 'detail': f'keep Store-Dept series with at least {MIN_LEN} weekly observations'},
    {'step': 'reindex_weekly', 'detail': "asfreq('W-FRI') per series, then interpolate + bfill + ffill Weekly_Sales"},
    {'step': 'holiday_flags', 'detail': 'IsHoliday cast to bool per series, missing weeks treated as non-holiday'},
    {'step': 'chronological_split', 'detail': f'last {VAL_LEN} weeks of each series held out as validation'},
    {'step': 'windowing', 'detail': f'sliding ({INPUT_LEN} -> {OUTPUT_LEN}) windows built from the fit portion only'},
    {'step': 'monitor_holdout', 'detail': 'last fit window of each series reserved as the early-stopping monitor set'},
    {'step': 'instance_normalization', 'detail': 'RevIN inside the model; no global scaler is fitted'},
]


with mlflow.start_run(run_name='PatchTST_Cleaning'):
    mlflow.set_tag('stage', 'cleaning')
    mlflow.log_params({
        'input_len': INPUT_LEN,
        'output_len': OUTPUT_LEN,
        'val_len': VAL_LEN,
        'min_series_len': MIN_LEN,
        'resample_freq': 'W-FRI',
        'gap_fill': 'interpolate + bfill + ffill',
        'scaler': 'RevIN (per-window, inside the model)',
        'n_preprocessing_steps': len(PREPROCESSING_STEPS),
    })
    mlflow.log_metrics({
        'train_rows': train.shape[0],
        'train_cols': train.shape[1],
        'train_missing_cells': int(train.isna().sum().sum()),
        'n_stores': int(train['Store'].nunique()),
        'n_depts': int(train['Dept'].nunique()),
        'n_store_dept_pairs': int(train[['Store', 'Dept']].drop_duplicates().shape[0]),
        'n_series_candidate': len(top_series),
        'n_series_kept': len(keys_kept),
        'n_series_dropped': len(top_series) - len(keys_kept),
        'train_windows': X_train.shape[0],
        'monitor_windows': X_monitor.shape[0],
    })
    mlflow.log_dict({'steps': PREPROCESSING_STEPS}, 'preprocessing/pipeline.json')
    mlflow.log_dict({'columns': train.columns.tolist()}, 'preprocessing/processed_columns.json')
    mlflow.log_input(
        mlflow.data.from_pandas(train, name='train_processed', targets='Weekly_Sales'),
        context='training',
    )


with mlflow.start_run(run_name='PatchTST_Training'):
    mlflow.set_tag('stage', 'training')
    mlflow.log_params({
        'patch_len': PATCH_LEN,
        'stride': STRIDE,
        'num_patches': NUM_PATCHES,
        'd_model': 64,
        'nhead': 4,
        'num_layers': 3,
        'dim_feedforward': 256,
        'dropout': 0.1,
        'loss': 'HuberLoss(delta=1.0)',
        'optimizer': 'AdamW',
        'learning_rate': 5e-4,
        'weight_decay': 1e-2,
        'scheduler': 'CosineAnnealingLR',
        'batch_size': 1024,
        'grad_clip_norm': 0.5,
        'max_epochs': N_EPOCHS,
        'early_stopping_patience': patience,
        'device': device,
    })

    for h in history:
        mlflow.log_metrics(
            {'train_loss': h['train_loss'], 'val_loss': h['val_loss'], 'lr': h['lr']},
            step=h['epoch'],
        )

    mlflow.log_metrics({
        'best_val_loss': best_val_loss,
        'best_epoch': best_epoch,
        'epochs_trained': epochs_trained,
        'n_parameters': sum(p.numel() for p in model_patchtst.parameters()),
    })

    with tempfile.TemporaryDirectory() as tmp:
        p = os.path.join(tmp, 'training_history.csv')
        pd.DataFrame(history).to_csv(p, index=False)
        mlflow.log_artifact(p, artifact_path='training')

with mlflow.start_run(run_name='PatchTST_Validation'):
    mlflow.set_tag('stage', 'validation')
    mlflow.log_params({
        'scheme': 'chronological holdout (no k-fold: the target is a time series)',
        'val_weeks': VAL_LEN,
        'forecast_strategy': f'roll the {OUTPUT_LEN}-week head forward {VAL_LEN // OUTPUT_LEN}x, feeding predictions back as context',
        'n_series': len(keys_kept),
        'val_points': len(all_true),
    })
    mlflow.log_metrics({
        'final_wmae': patchtst_wmae,
        'val_wmae_holiday': wmae(all_true[all_holiday], all_pred[all_holiday], all_holiday[all_holiday]),
        'val_wmae_non_holiday': wmae(all_true[~all_holiday], all_pred[~all_holiday], all_holiday[~all_holiday]),
    })

with mlflow.start_run(run_name='PatchTST_Final_Model') as final_run:
    mlflow.set_tags({'stage': 'final_model', 'algorithm': 'patchtst'})
    mlflow.log_params({
        'input_len': INPUT_LEN,
        'output_len': OUTPUT_LEN,
        'patch_len': PATCH_LEN,
        'num_patches': NUM_PATCHES,
        'n_series': len(keys_kept),
        'best_epoch': best_epoch,
    })
    mlflow.log_metrics({
        'final_wmae': patchtst_wmae,
        'best_val_loss': best_val_loss,
    })


    model_info = mlflow.pytorch.log_model(
        model_patchtst,
        artifact_path='model',

        pickle_module=cloudpickle,
        input_example=X_monitor[:5],
    )
    mlflow.log_artifact('models/patchtst_best.pkl', artifact_path='estimator')
    print('PatchTST final run:', final_run.info.run_id)

/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run PatchTST_Cleaning at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/7/runs/0e71bd98ee9b4ef29aea2a9b9f5d1913
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/7
🏃 View run PatchTST_Training at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/7/runs/3d9b4f7ec0f14eae99f97a71c528f616
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/7
🏃 View run PatchTST_Validation at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/7/runs/34e08830ff384feea30ea48b09fa25cc
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/7


2026/07/12 12:06:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 12:06:56 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/07/12 12:07:08 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpfr57oo2k/model/data, flavor: pytorch). Fall back to return ['torch==2.11.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
/usr/local/lib/python3.12/dist-packages/torch/export/pt2_archive/_package.py:790: UserWarning: The given buffer is not writable, and PyTorch does not support non-writable tensors. This means you can write to the underlyin

PatchTST final run: 2bd56493058b455eb2a2f7ca43a64f34
🏃 View run PatchTST_Final_Model at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/7/runs/2bd56493058b455eb2a2f7ca43a64f34
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/7
